# Lab 8.11 &mdash; Assemble the Customer-Service Chatbot

**Level:** Advanced &nbsp;|&nbsp; **Est. time:** 35 min &nbsp;|&nbsp; **Day 4 &middot; Module 8 &mdash; Multi-Agent Collaboration &amp; Decision Making**

### What you'll do
- Build billing & tech specialists as real agent executors
- Route a message and run only the matching specialists
- Synthesise one reply, flagged needs_approval for the refund

> **How this lab works (experiential flow):** read the **Concept**, run the **Demo** to see it work, then complete **Your Turn** by replacing every `___` placeholder. Run the **grader** cell at the end &mdash; it prints `[PASS]` / `[FAIL]` / `[TODO]` and a final `Score`. Aim for a full score.

> **Framework note:** the graded steps are **offline &amp; deterministic** (pure Python stdlib); the agent-assembly labs reuse the **LangChain-shaped shim** from Modules 6&ndash;7. Advanced labs end with an **optional** cell that runs the **real** library. You are building the **multi-agent customer-service chatbot** &mdash; the client's Lab 4.2.

**Reference:** [Module 8 slides &mdash; The multi-agent customer-service chatbot](../../presentation/day4-module8-multi-agent-collaboration.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html) &nbsp;&middot;&nbsp; [All Module 8 labs](./index.html)

In [ ]:
# Setup -- run me first
import os
WORK = "/tmp/biaa-lab-08-11"
os.makedirs(WORK, exist_ok=True)
print("Working dir:", WORK)

## Concept
Now assemble the chatbot from Modules 6&ndash;7 pieces (deck slides 15&ndash;17): each specialist is a
`create_react_agent` + `AgentExecutor` with its **own** prompt and **own** small tool set. The
**supervisor** routes the message; the matching specialists run (each grounded by its tool); their
findings are **synthesised** into one reply. Because a refund is **irreversible**, the reply is flagged
**`needs_approval`** &mdash; draft-not-send, now at the team level.

In [ ]:
# --- LangChain-SHAPED shim: a tool has .name, .description (from the docstring), .args, .invoke() ---
import inspect
class Tool:
    def __init__(self, fn, name=None, description=None):
        self.fn = fn
        self.name = name or fn.__name__
        self.description = (description or inspect.getdoc(fn) or "").strip()
        self.args = list(inspect.signature(fn).parameters)
    def invoke(self, value):
        return self.fn(**value) if isinstance(value, dict) else self.fn(value)
    def __repr__(self):
        return "Tool(name=%r)" % self.name
def tool(fn):
    """The @tool decorator -- wrap a plain function into a Tool (mirrors langchain_core.tools.tool)."""
    return Tool(fn)

class AIMessage:
    def __init__(self, content): self.content = content
class FakeChatModel:
    """Deterministic stand-in for ChatOllama / ChatGroq: replays a scripted list of replies.
    Real code: from langchain_ollama import ChatOllama; model = ChatOllama(model="llama3.2:1b").
    Like the real thing, .invoke(prompt) returns a message whose text is in .content."""
    def __init__(self, script): self.script = list(script); self.i = 0
    def invoke(self, prompt):
        reply = self.script[min(self.i, len(self.script) - 1)]; self.i += 1
        return AIMessage(reply)

class PromptTemplate:
    """Mirrors LangChain: PromptTemplate.from_template(...).format(input=..., ...)."""
    def __init__(self, template): self.template = template
    @classmethod
    def from_template(cls, template): return cls(template)
    def format(self, **kw):
        s = self.template
        for k, v in kw.items():
            s = s.replace("{" + k + "}", str(v))
        return s

def create_react_agent(model, tools, prompt):
    """Bind model + tools + prompt into a ReAct agent (mirrors langchain.agents.create_react_agent)."""
    return {"model": model, "tools": {t.name: t for t in tools}, "prompt": prompt}
def parse_react(text):
    """Turn the model's ReAct text into ('final', answer) or ('action', name, input)."""
    action = inp = None
    for line in text.splitlines():
        s = line.strip()
        if s.startswith("Final Answer:"): return ("final", s.split(":", 1)[1].strip())
        if s.startswith("Action Input:"): inp = s.split(":", 1)[1].strip()
        elif s.startswith("Action:"):      action = s.split(":", 1)[1].strip()
    return ("action", action, inp)
class AgentExecutor:
    """Runs the loop: ask model -> parse -> run tool -> observe -> repeat, capped by max_iterations
    (mirrors langchain.agents.AgentExecutor). verbose=True prints the ReAct trace."""
    def __init__(self, agent, tools=None, verbose=False, max_iterations=6):
        self.agent = agent
        self.tools = agent["tools"] if tools is None else {t.name: t for t in tools}
        self.model = agent["model"]; self.prompt = agent["prompt"]
        self.verbose = verbose; self.max_iterations = max_iterations
    def invoke(self, inputs):
        scratch, steps = "", []
        for _ in range(self.max_iterations):
            text = self.model.invoke(self.prompt.format(input=inputs["input"], scratchpad=scratch)).content
            if self.verbose: print(text)
            parsed = parse_react(text)
            if parsed[0] == "final":
                return {"output": parsed[1], "intermediate_steps": steps}
            name, arg = parsed[1], parsed[2]
            obs = self.tools[name].invoke(arg) if name in self.tools else ("unknown tool: %s" % name)
            if self.verbose: print("Observation:", obs)
            steps.append((name, arg, obs)); scratch += text + "\nObservation: " + str(obs) + "\n"
        return {"output": None, "intermediate_steps": steps}

# The chatbot's context sources: invoices (order 4471 has a DUPLICATE charge) and known issues.
INVOICES = {
    "4471": [{"amount": 50, "date": "Jul 01"}, {"amount": 50, "date": "Jul 01"}],
    "5090": [{"amount": 30, "date": "Jul 02"}],
}
KNOWN_ISSUES = {
    "crash": {"bug": "BUG-231", "fix": "update to v4.2"},
    "login": {"bug": "BUG-118", "fix": "reset your password"},
}

@tool
def lookup_invoice(order_id):
    """Look up the charges on an order by id."""
    return INVOICES.get(order_id, [])
@tool
def known_issues(symptom):
    """Look up a known technical issue by symptom keyword."""
    for k, v in KNOWN_ISSUES.items():
        if k in symptom.lower():
            return v
    return {}
def synthesize(findings):
    return " ".join(findings[k] for k in sorted(findings)) if findings else "No findings."
print("tools & synthesise ready:", lookup_invoice.name, "&", known_issues.name)

## Your Turn
Complete `run_specialist` (build & run one specialist agent) and `handle_message` (route, run,
synthesise, flag).

In [ ]:
BILLING_SCRIPT = [
    "Thought: check the charges.\nAction: lookup_invoice\nAction Input: 4471",
    "Thought: two charges -- a duplicate.\nFinal Answer: On billing: duplicate charge on 4471, refund warranted.",
]
TECH_SCRIPT = [
    "Thought: look up the symptom.\nAction: known_issues\nAction Input: crash",
    "Thought: known bug.\nFinal Answer: On the app: matches BUG-231, fix is update to v4.2.",
]

def run_specialist(script, tools):
    model  = FakeChatModel(script)
    prompt = PromptTemplate.from_template("Message: {input}\n{scratchpad}")
    agent  = create_react_agent(model, ___, prompt)   # TODO: this specialist's own tools
    return AgentExecutor(agent, max_iterations=6)

def handle_message(message):
    m = message.lower()
    findings = {}
    if any(k in m for k in ("charg", "refund", "invoice", "billed")):
        findings["billing"] = ___   # TODO: run the billing specialist, take its "output"
    if any(k in m for k in ("crash", "bug", "login", "broken")):
        findings["tech"] = run_specialist(TECH_SCRIPT, [known_issues]).invoke({"input": message})["output"]
    return {"reply": synthesize(findings), "status": ___, "agents": sorted(findings)}   # TODO: needs_approval

try:
    out = handle_message('I was charged twice for 4471 and the app keeps crashing')
    print('agents:', out['agents'])
    print('status:', out['status'])
    print('reply :', out['reply'])
except Exception as e:
    print('Fill the blanks, then re-run.', type(e).__name__)

In [ ]:
# === Auto-grader: run after filling the blanks above ===
_results = []
def _rec(label, status, extra=""):
    _results.append(status); print(f"[{status}] {label}" + (f" -- {extra}" if extra else ""))
def expect(label, got, want):
    if got == "___" or got is None: _rec(label, "TODO")
    elif got == want: _rec(label, "PASS")
    else: _rec(label, "FAIL", f"got {got!r}")
def expect_true(label, fn):
    try: _rec(label, "PASS" if fn() else "FAIL")
    except Exception as e: _rec(label, "TODO", type(e).__name__)

expect_true("the billing specialist reports the duplicate", lambda: "billing" in handle_message("I was charged twice for 4471")["reply"].lower() or "duplicate" in handle_message("I was charged twice for 4471")["reply"].lower())
expect_true("a two-intent message engages BOTH specialists", lambda: handle_message("charged twice for 4471 and the app keeps crashing")["agents"] == ["billing", "tech"])
expect_true("the reply synthesises both findings", lambda: (lambda r: "refund" in r.lower() and "bug-231" in r.lower())(handle_message("charged twice and crashing")["reply"]))
expect_true("a pure tech message engages only tech", lambda: handle_message("the app keeps crashing")["agents"] == ["tech"])
expect_true("the reply is needs_approval (refund is irreversible)", lambda: handle_message("charged twice")["status"] == "needs_approval")

_p = _results.count("PASS")
print(f"\nScore: {_p}/{len(_results)}")
print("All checks passed -- lab complete!" if _p == len(_results) else "Keep going: fill the blanks marked ___ and re-run.")

## Optional &mdash; run this against the REAL LangChain / LangGraph (not graded)
Swap the scripted specialists for REAL LangChain agents behind a supervisor (LangGraph, or two agents by hand). Safe to skip &mdash; it needs `pip install langchain langchain-ollama langgraph` (then
`ollama run llama3.2:1b`) or `langchain-groq` with a `GROQ_API_KEY`. If a package, model or key is
missing the cell prints a friendly note and moves on.
**The graded steps above are offline &amp; deterministic, so the lab always verifies without a model.**

In [ ]:
try:
    from langchain_ollama import ChatOllama
    llm = ChatOllama(model="llama3.2:1b")
    billing = llm.invoke("You are a BILLING agent. In one line, handle: charged twice on order 4471.").content
    tech    = llm.invoke("You are a TECH agent. In one line, handle: the app keeps crashing.").content
    print("REAL billing agent:", billing)
    print("REAL tech agent   :", tech)
    print("In production: give each its own tools via create_react_agent, and wire them behind a")
    print("LangGraph StateGraph supervisor -- then synthesise, and gate the refund on human approval.")
except Exception as e:
    print("No local LLM available -- skipping (pip install langchain langchain-ollama langgraph +")
    print("`ollama run llama3.2:1b`, or langchain-groq with GROQ_API_KEY):", type(e).__name__)
    print("The offline chatbot above already routed to both specialists and synthesised one needs_approval reply.")

---
### Key takeaway
Same executors as Module 6, now a TEAM: a supervisor routes, specialists gather with their own tools, a synthesiser makes one reply -- and the refund waits for a human. Next: run it over a whole suite.

[&#8592; All Module 8 labs](./index.html) &nbsp;&middot;&nbsp; [Module 8 slides](../../presentation/day4-module8-multi-agent-collaboration.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html)

<sub>&copy; 2026 Gheware DevOps &amp; Agentic AI &middot; Building Intelligent AI Agents &middot; devops.gheware.com &middot; Trainer: Rajesh Gheware</sub>